# OCR y extracción estructurada sobre un documento público real

**Data Science con Python — Universidad del Pacífico, 2026-II**
Semana 10: *Document AI y voz*. Práctica del miércoles 21 de octubre, 7:30–9:20 am.

---

## Qué vamos a hacer

Tomar las **bases de una licitación pública peruana**, que llegan como un PDF escaneado, y
convertirlas en datos que un programa pueda usar.

El documento es real: Licitación Pública de Obras N° 05-2026-C-MPI de la Municipalidad
Provincial de Ilo, descargada de SEACE. Tiene 57 hojas y cuatro problemas a la vez:

| Problema | Qué significa |
|---|---|
| Cero texto seleccionable | Es una foto. No se puede copiar nada |
| Dos páginas por hoja | 57 hojas contienen unas 114 páginas del documento |
| Todo girado 90° | Hay que inclinar la cabeza para leerlo |
| Firmas encima del texto | Ruido de lapicero azul sobre las letras |

## La idea única de la sesión

> **Conocer tu hardware decide qué modelo puedes correr.
> Y arreglar la imagen rinde más que traer un modelo más grande.**

Todo lo demás son consecuencias de esa frase.

---

## Los tres carriles

Elige uno. **Los tres corren exactamente el mismo código.** Lo único que cambia es la
máquina, y eso en sí mismo es la lección.

| Carril | Para quién | Qué usa |
|---|---|---|
| **A — Colab** | la mayoría | GPU T4 gratuita de Google |
| **B — Local CPU** | quien no tenga tarjeta NVIDIA | tu procesador |
| **C — Local GPU** | quien tenga tarjeta NVIDIA | tu GPU |

Si en algún momento algo falla, pasa al carril A y sigue. Nadie se queda sin hacer OCR hoy.

---

# Bloque 0 — Preparar el entorno

## Dos advertencias que te van a ahorrar media hora

**1. NO instales el CUDA Toolkit.**
La documentación oficial de PaddlePaddle lo dice: el paquete gestiona CUDA y cuDNN solo;
basta con tener el driver de NVIDIA.

> *Analogía:* el CUDA Toolkit es la **fábrica** de juguetes. El paquete de Paddle ya trae
> los **juguetes hechos**. Tú quieres jugar, no fabricar. Instalar una fábrica de 3 GB para
> jugar con algo que ya viene en la caja es el error clásico.

**2. La versión de Python importa.**
PaddlePaddle soporta Python 3.9 a 3.13. Si tienes 3.14, el error no te va a decir "versión
no soportada": te va a decir que no encuentra el paquete, y vas a buscar el problema donde
no está. Crea un entorno con Python 3.12.

In [ ]:
# Configuración de la sesión. Esto es lo único que vas a tocar.
CONFIG = {
    "pdf": "BASE-ESTANDAR-ILO-SEACE-OCR.pdf",  # nombre del archivo
    "dpi_fisico": 425,       # la hoja trae 2 paginas; 425 deja cada una cerca de 300 DPI
    "dos_por_pagina": True,  # partir cada hoja en dos paginas logicas
    "device": "gpu",         # "gpu" o "cpu". OJO: en PaddleOCR 3.x es device, NO use_gpu
    "idioma": "es",
    "hoja_demo": 1,          # hoja fisica que usaremos de ejemplo (0 = la primera)
    "bench_calentamiento": 2,
    "bench_medidas": 6,
}

INSTALAR = True  # ponlo en False si ya instalaste todo

import sys, os, subprocess, platform
EN_COLAB = "google.colab" in sys.modules
print("Python     :", platform.python_version())
print("Entorno    :", "Google Colab" if EN_COLAB else "Local")
print("Plataforma :", platform.system(), platform.release())

if sys.version_info >= (3, 14):
    print()
    print("[ALTO] Python 3.14 o superior no esta soportado por PaddlePaddle (3.9 a 3.13).")
    print("       Crea un entorno con Python 3.12 antes de seguir.")

In [ ]:
# Instalacion. En Colab tarda unos minutos: es descarga pura, no te quedes mirando la barra.
if INSTALAR:
    hay_nvidia = subprocess.run("nvidia-smi", shell=True,
                                stdout=subprocess.DEVNULL,
                                stderr=subprocess.DEVNULL).returncode == 0
    print("GPU NVIDIA detectada:", hay_nvidia)

    if hay_nvidia:
        # cu126 es el indice mas probado. Con driver moderno tambien sirven cu129 y cu130.
        %pip install -q paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
    else:
        %pip install -q paddlepaddle==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
        CONFIG["device"] = "cpu"

    # PP-OCRv6 necesita paddleocr 3.7 o superior.
    %pip install -q "paddleocr>=3.7" pypdfium2 pypdf pandas pillow
    print("\nListo. Si Colab te pide reiniciar el entorno, reinicia y vuelve a correr desde aqui.")

---

# El doctor: ¿la GPU está realmente encendida?

Esta es la celda más importante del notebook y la razón por la que existe esta clase.

**PaddlePaddle no lanza error cuando la GPU falla.** Se cae a CPU en silencio y sigue
funcionando. Tú crees que tienes aceleración y no la tienes.

> *Analogía:* tienes una bicicleta y un auto en la cochera. Le dices al chofer "llévame".
> Si la llave del auto no está colgada en el llavero, **no te dice nada**: te lleva en
> bicicleta. Llegas tarde y nunca te enteras de que el auto existía.

El doctor revisa seis eslabones. Los cinco primeros pueden dar verde y aun así el OCR
correr en CPU, porque falta el parámetro `device`. **Solo el sexto no se puede falsificar.**

In [ ]:
import re, time, json, shutil, textwrap
from pathlib import Path

def vram_mb():
    '''VRAM ocupada en MB segun nvidia-smi. Devuelve None si no hay GPU.'''
    try:
        s = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            text=True, stderr=subprocess.DEVNULL)
        return float(s.strip().splitlines()[0])
    except Exception:
        return None

def doctor(verbose=True):
    '''Seis chequeos. Devuelve el dispositivo que realmente se puede usar.'''
    lineas, device_ok = [], False

    # 1. Driver
    try:
        s = subprocess.check_output(["nvidia-smi"], text=True, stderr=subprocess.DEVNULL)
        gpu = re.search(r"(NVIDIA[^|]*?)\s{2,}", s)
        drv = re.search(r"Driver Version:\s*([\d.]+)", s)
        lineas.append(("1. Driver NVIDIA", "OK",
                       f"{gpu.group(1).strip() if gpu else '?'} / driver {drv.group(1) if drv else '?'}"))
    except Exception:
        lineas.append(("1. Driver NVIDIA", "FALLA", "nvidia-smi no responde: no hay GPU usable"))

    # 2. Un solo wheel de Paddle instalado
    try:
        pl = subprocess.check_output([sys.executable, "-m", "pip", "list"], text=True)
        cpu_wheel = bool(re.search(r"^paddlepaddle\s", pl, re.M))
        gpu_wheel = bool(re.search(r"^paddlepaddle-gpu\s", pl, re.M))
        if cpu_wheel and gpu_wheel:
            lineas.append(("2. Wheel unico", "FALLA",
                           "estan instalados paddlepaddle Y paddlepaddle-gpu: todo corre en CPU en silencio"))
        elif gpu_wheel:
            lineas.append(("2. Wheel unico", "OK", "solo paddlepaddle-gpu"))
        else:
            lineas.append(("2. Wheel unico", "AVISO", "solo la version CPU instalada"))
    except Exception as e:
        lineas.append(("2. Wheel unico", "AVISO", f"no se pudo verificar ({e})"))

    # 3 a 5. Lo que dice Paddle de si mismo
    try:
        import paddle
        compilado = paddle.is_compiled_with_cuda()
        lineas.append(("3. Compilado con CUDA", "OK" if compilado else "FALLA", str(compilado)))
        dev = paddle.device.get_device()
        lineas.append(("4. Dispositivo activo", "OK" if dev.startswith("gpu") else "AVISO", dev))
        try:
            import io, contextlib
            buf = io.StringIO()
            with contextlib.redirect_stdout(buf):
                paddle.utils.run_check()
            texto = buf.getvalue()
            ok = "success" in texto.lower() or "exito" in texto.lower()
            lineas.append(("5. run_check()", "OK" if ok else "AVISO", texto.strip().splitlines()[-1][:70]))
        except Exception as e:
            lineas.append(("5. run_check()", "FALLA", str(e)[:70]))
        device_ok = compilado and dev.startswith("gpu")
    except Exception as e:
        lineas.append(("3-5. Paddle", "FALLA", f"no se pudo importar paddle: {e}"))

    # 6. La unica prueba que no se puede falsificar
    base = vram_mb()
    if base is None:
        lineas.append(("6. Prueba real de VRAM", "AVISO", "sin GPU, no aplica"))
    else:
        lineas.append(("6. Prueba real de VRAM", "PENDIENTE",
                       f"linea base {base:.0f} MB; se mide al cargar el modelo"))

    if verbose:
        print("=" * 78)
        for nombre, estado, detalle in lineas:
            print(f"[{estado:^9}] {nombre:<26} {detalle}")
        print("=" * 78)
        print("Dispositivo recomendado:", "gpu" if device_ok else "cpu")
        if base is not None:
            print("Nota: cierra el navegador antes de medir. Cada pestaña ocupa VRAM.")
    return "gpu" if device_ok else "cpu"

DEVICE = doctor()
CONFIG["device"] = DEVICE

### Cómo leer el resultado

Si el chequeo 3 o el 4 salen mal, hay **exactamente tres causas posibles**, en este orden:

1. Instalaste `paddlepaddle` (CPU) en lugar de `paddlepaddle-gpu`.
2. Están los dos instalados. Desinstala el de CPU: `pip uninstall -y paddlepaddle`.
3. El entorno virtual activo no es el que crees.

No hay una cuarta causa. No pierdas tiempo buscándola.

---

# Conseguir el documento

En **Colab**: ejecuta la celda y sube el PDF cuando te lo pida.
En **local**: deja el PDF en la misma carpeta que este notebook.

Si no tienes el PDF a la mano, la celda genera un documento de práctica con los mismos
cuatro problemas (escaneado, girado, dos por hoja) para que puedas seguir igual.

In [ ]:
from PIL import Image, ImageDraw, ImageFont

RUTA_PDF = None
for cand in [CONFIG["pdf"], f"data/raw/{CONFIG['pdf']}", f"../data/raw/{CONFIG['pdf']}"]:
    if Path(cand).exists():
        RUTA_PDF = Path(cand); break

if RUTA_PDF is None and EN_COLAB:
    from google.colab import files
    print("Sube el PDF de las bases (o cualquier PDF escaneado):")
    subidos = files.upload()
    if subidos:
        RUTA_PDF = Path(list(subidos.keys())[0])

if RUTA_PDF is None:
    # Plan B: fabricamos un documento con los mismos defectos, para no quedarnos sin clase.
    print("No hay PDF. Genero uno de practica con los mismos cuatro problemas.\n")
    import pypdfium2  # solo para confirmar que esta instalado

    def hoja_falsa():
        '''Una hoja A4 con dos paginas logicas, ambas giradas 90 grados.'''
        A4 = (2480, 3508)  # A4 a 300 DPI
        hoja = Image.new("RGB", A4, "white")
        try:
            f_t = ImageFont.truetype("DejaVuSans-Bold.ttf", 54)
            f_n = ImageFont.truetype("DejaVuSans.ttf", 38)
        except Exception:
            f_t = f_n = ImageFont.load_default()

        textos = [
            ["MUNICIPALIDAD PROVINCIAL DE DEMO",
             "LICITACION PUBLICA DE OBRAS N 05-2026-C-DEM",
             "",
             "CONTRATACION PARA LA EJECUCION DE LA OBRA:",
             "MEJORAMIENTO DE VIAS LOCALES DEL SECTOR II,",
             "DISTRITO DE DEMO - DEPARTAMENTO DE MOQUEGUA."],
            ["CAPITULO I - ASPECTOS GENERALES",
             "",
             "1.1 REFERENCIAS",
             "Cuando en el presente documento se mencione la",
             "palabra Ley, se entiende que se esta haciendo",
             "referencia a la Ley General de Contrataciones."],
        ]
        for i, bloque in enumerate(textos):
            # cada pagina logica es media hoja, en horizontal, y luego se gira
            pag = Image.new("RGB", (A4[1] // 2, A4[0]), "white")
            d = ImageDraw.Draw(pag)
            y = 180
            for j, linea in enumerate(bloque):
                d.text((140, y), linea, fill="black", font=f_t if j == 0 else f_n)
                y += 90 if j == 0 else 70
            pag = pag.rotate(90, expand=True)          # el defecto: girada
            pag = pag.resize((A4[0], A4[1] // 2))
            hoja.paste(pag, (0, 0 if i == 1 else A4[1] // 2))  # la pagina 1 va abajo
        return hoja

    hojas = [hoja_falsa() for _ in range(3)]
    RUTA_PDF = Path("documento_de_practica.pdf")
    hojas[0].save(RUTA_PDF, save_all=True, append_images=hojas[1:], resolution=300.0)

print("Documento:", RUTA_PDF, f"({RUTA_PDF.stat().st_size/1e6:.1f} MB)")

---

# Bloque 1 — Triaje: la regla de oro

Antes de hacer OCR hay que preguntarse si hace falta. Existen **dos tipos de PDF**:

| PDF con capa de texto | PDF escaneado (imagen) |
|---|---|
| El texto es seleccionable | Es una foto: no hay texto |
| Se extrae gratis y al instante | Requiere OCR: lento y caro |

> **Regla de oro: usa OCR solo en las páginas que son imagen.**
> En el resto, extracción directa.

> *Analogía:* el triaje es la enfermera que decide quién pasa a emergencia. No todos los
> pacientes necesitan cirugía. El OCR es la cirugía: cara y lenta. Solo se opera a quien lo
> necesita.

In [ ]:
from pypdf import PdfReader
import pandas as pd

def triaje(ruta_pdf, umbral_caracteres=50):
    '''Clasifica cada hoja en 'texto' o 'imagen'.'''
    lector = PdfReader(str(ruta_pdf))
    filas = []
    for i, pagina in enumerate(lector.pages):
        try:
            txt = pagina.extract_text() or ""
        except Exception:
            txt = ""
        n = len(txt.strip())
        # cuantas imagenes embebidas trae la hoja
        n_img = 0
        try:
            recursos = pagina.get("/Resources") or {}
            xo = recursos.get("/XObject")
            if xo:
                xo = xo.get_object()
                n_img = sum(1 for k in xo if xo[k].get_object().get("/Subtype") == "/Image")
        except Exception:
            pass
        filas.append({"hoja": i, "caracteres": n, "imagenes_embebidas": n_img,
                      "clase": "texto" if n >= umbral_caracteres else "imagen"})
    return pd.DataFrame(filas)

tr = triaje(RUTA_PDF)
n_img = (tr["clase"] == "imagen").sum()
print(f"{len(tr)} hojas: {len(tr)-n_img} con texto, {n_img} imagen "
      f"-> {100*n_img/len(tr):.0f}% requiere OCR\n")
display(tr.head(10))

### Lee el resultado antes de seguir

Sobre el documento real el resultado es **extremo y limpio: 0 hojas con texto, 57 imagen**.
Cero por ciento aprovechable sin OCR.

Fíjate también en la columna `imagenes_embebidas`: hay entre **9 y 63 imágenes por hoja**.
Eso no significa que haya 63 fotos; significa que el escáner guardó cada hoja **partida en
mosaicos**.

> *Analogía:* el PDF no guarda la foto de la hoja, guarda el **rompecabezas desarmado**. Si
> le pides "las imágenes", te da las piezas sueltas. Hay que pedirle la foto del
> rompecabezas armado, y eso se llama **rasterizar**.

**Consecuencia práctica:** nunca extraigas las imágenes embebidas de un PDF escaneado.
Renderiza la página completa. Es el error más común en este tema.

### Pregunta para el salón

Si de un documento de 500 páginas solo 20 son escaneadas, ¿cuánto te ahorra el triaje?
Calcúlalo con los segundos por página que vas a medir en el Bloque 4.

---

# Bloque 2 — Preparar la imagen

Aquí se gana la pelea. Tres pasos, **en este orden**:

1. **Rasterizar** la hoja a un bitmap con un DPI elegido.
2. **Partir** la hoja en sus dos páginas lógicas.
3. **Enderezar** la página.

Partir después del OCR no sirve: el modelo ya mezcló las dos páginas.

## Por qué el DPI es un parámetro y no una constante

Como cada página lógica ocupa **media hoja A4**, está reducida a un factor lineal de
aproximadamente 0.707. Para que la página lógica quede en 300 DPI efectivos, la hoja física
debe renderizarse a unos **425 DPI**.

> *Analogía:* si fotocopias dos hojas en una, las letras salen más chiquitas. Para leerlas
> igual de bien necesitas una lupa más fuerte. El DPI es la lupa.

In [ ]:
import pypdfium2 as pdfium

def rasterizar(ruta_pdf, indice_hoja, dpi):
    '''Renderiza una hoja completa del PDF como imagen PIL.'''
    doc = pdfium.PdfDocument(str(ruta_pdf))
    pagina = doc[indice_hoja]
    img = pagina.render(scale=dpi / 72).to_pil()
    doc.close()
    return img

hoja = rasterizar(RUTA_PDF, CONFIG["hoja_demo"], CONFIG["dpi_fisico"])
print(f"Hoja {CONFIG['hoja_demo']} rasterizada a {CONFIG['dpi_fisico']} DPI: {hoja.size} pixeles")
hoja.resize((hoja.width // 4, hoja.height // 4))

Mírala bien: el texto corre **de costado**. Y hay **dos páginas** del documento en esa sola
hoja.

Ahora la partimos. En este documento la página de número más bajo va **abajo** y la de
número más alto va **arriba**, así que el orden de lectura es: mitad inferior primero.

In [ ]:
def partir_en_dos(img):
    '''Parte una hoja A4 en sus dos paginas logicas. Devuelve [inferior, superior].'''
    w, h = img.size
    superior = img.crop((0, 0, w, h // 2))
    inferior = img.crop((0, h // 2, w, h))
    return [inferior, superior]   # la pagina de numero menor va abajo

mitades = partir_en_dos(hoja)
print("Dos paginas logicas:", [m.size for m in mitades])
mitades[0].resize((mitades[0].width // 3, mitades[0].height // 3))

## Enderezar: pruébalo tú

Todavía está de costado. ¿Hay que girar 90°, 180° o 270°?

En lugar de adivinar, la siguiente celda te muestra las cuatro opciones. **Elige mirando.**
Esto es exactamente lo que hace el clasificador de orientación de PaddleOCR, solo que
automáticamente.

In [ ]:
import matplotlib.pyplot as plt

pagina = mitades[0]
fig, ejes = plt.subplots(1, 4, figsize=(16, 6))
for eje, ang in zip(ejes, [0, 90, 180, 270]):
    eje.imshow(pagina.rotate(ang, expand=True))
    eje.set_title(f"{ang} grados"); eje.axis("off")
plt.tight_layout(); plt.show()

ROTACION = 270   # cambia este numero al angulo que se lee bien
                 # (para el documento de Ilo la respuesta es 270)
pagina_lista = pagina.rotate(ROTACION, expand=True)
print("Fijate en un detalle: el documento original dice ASENTAMIENTO 'HUAMANO'.")
print("Es una errata de la Municipalidad, no del OCR. El OCR debe copiarla tal cual.")
Path("procesadas").mkdir(exist_ok=True)
RUTA_PAGINA = Path("procesadas/pagina_demo.png")
pagina_lista.save(RUTA_PAGINA)
print("Pagina lista:", RUTA_PAGINA, pagina_lista.size)

---

# Bloque 3 — El primer OCR

Un detalle que casi nadie nota al principio: **el OCR no devuelve texto**. Devuelve tres
cosas, y las tres valen igual.

| Campo | Qué es | Para qué sirve |
|---|---|---|
| `texto` | la cadena reconocida | lo obvio |
| `caja` | cuatro puntos sobre la imagen | permite resaltar en el original lo que la IA afirma |
| `confianza` | número entre 0 y 1 | permite decidir qué revisa un humano |

> *Analogía:* no es un lector que te cuenta el libro. Es un lector que además te **señala
> con el dedo** en qué parte de la página está cada frase, y te dice **qué tan seguro está**
> de cada una.

In [ ]:
from paddleocr import PaddleOCR

def construir_motor(device="gpu", idioma="es", orientacion=True):
    '''Construye PaddleOCR tolerando diferencias entre versiones 3.x.'''
    # En CPU hay que apagar oneDNN. Verificado sobre paddlepaddle-gpu 3.3.0 en
    # Windows: con oneDNN activo, CADA inferencia en CPU muere con
    #     NotImplementedError: ConvertPirAttribute2RuntimeAttribute not support
    # Es un bug del ejecutor PIR, no del documento ni del modelo. Sin esto, todo
    # el carril B (los estudiantes sin GPU) se queda sin clase.
    extra = {"enable_mkldnn": False} if device == "cpu" else {}
    if extra:
        print("CPU detectada: se apaga oneDNN (enable_mkldnn=False).")

    intentos = [
        dict(ocr_version="PP-OCRv6", lang=idioma, device=device,
             use_doc_orientation_classify=orientacion,
             use_doc_unwarping=False, use_textline_orientation=orientacion, **extra),
        dict(lang=idioma, device=device,
             use_doc_orientation_classify=orientacion,
             use_doc_unwarping=False, use_textline_orientation=orientacion, **extra),
        dict(lang=idioma, device=device, **extra),
        dict(device=device, **extra),
    ]
    for i, kw in enumerate(intentos, 1):
        try:
            motor = PaddleOCR(**kw)
            print(f"Motor construido (intento {i}) con: {sorted(kw)}")
            return motor
        except Exception as e:
            print(f"  intento {i} fallo: {str(e)[:90]}")
    raise RuntimeError("No se pudo construir PaddleOCR con ninguna combinacion")

# OJO: se construye UNA sola vez y se reutiliza.
# Reconstruirlo por pagina recarga el modelo y arruina cualquier medicion.
antes = vram_mb()
MOTOR = construir_motor(device=CONFIG["device"], idioma=CONFIG["idioma"])
despues = vram_mb()
if antes is not None and despues is not None:
    print(f"\n[Chequeo 6 del doctor] VRAM: {antes:.0f} -> {despues:.0f} MB "
          f"(delta {despues-antes:+.0f} MB)")
    print("Si el delta es practicamente cero, el modelo NO esta en la GPU.")

In [ ]:
def leer_resultado(res):
    '''Normaliza la salida de PaddleOCR 3.x a (textos, confianzas, cajas).'''
    obj = res
    for clave in ("res",):          # algunas versiones anidan bajo 'res'
        try:
            if clave in obj:
                obj = obj[clave]
        except Exception:
            pass

    def tomar(*claves):
        for k in claves:
            try:
                v = obj[k]
            except Exception:
                v = getattr(obj, k, None)
            if v is not None:
                return list(v)
        return []

    return (tomar("rec_texts"), tomar("rec_scores"),
            tomar("rec_polys", "dt_polys", "rec_boxes"))

def ocr_pagina(motor, ruta_img):
    '''Corre OCR sobre una imagen y devuelve un DataFrame.'''
    try:
        salida = motor.predict(str(ruta_img))
    except AttributeError:
        salida = motor.ocr(str(ruta_img))
    textos, scores, cajas = [], [], []
    for r in (salida or []):
        t, s, c = leer_resultado(r)
        textos += t; scores += s; cajas += c
    return pd.DataFrame({"texto": textos,
                         "confianza": scores,
                         "caja": cajas[:len(textos)] if cajas else [None]*len(textos)})

t0 = time.perf_counter()
df = ocr_pagina(MOTOR, RUTA_PAGINA)
print(f"{len(df)} lineas reconocidas en {time.perf_counter()-t0:.2f} s\n")
display(df[["texto", "confianza"]].head(20))
print("\nConfianza mediana:", round(df["confianza"].median(), 3) if len(df) else "n/a")
print("Lineas por debajo de 0.80:", int((df["confianza"] < 0.80).sum()))

In [ ]:
# Dibujamos las cajas sobre la imagen: esto es lo que convierte
# "confia en la IA" en "verifica en dos segundos".
img_cajas = pagina_lista.convert("RGB").copy()
dibujo = ImageDraw.Draw(img_cajas)
for _, fila in df.iterrows():
    if fila["caja"] is None:
        continue
    puntos = [(float(x), float(y)) for x, y in list(fila["caja"])]
    color = "green" if fila["confianza"] >= 0.80 else "red"
    dibujo.line(puntos + [puntos[0]], fill=color, width=6)

print("Verde: confianza >= 0.80   |   Rojo: por debajo, conviene revision humana")
img_cajas.resize((img_cajas.width // 3, img_cajas.height // 3))

### La idea más valiosa de la clase sale de aquí

La columna de confianza permite **escalar por costo**: corres el modelo barato sobre todo, y
solo reenvías al modelo caro las páginas donde el barato dudó.

> *Analogía:* el profesor corrige rápido todos los exámenes y solo vuelve a revisar con
> calma los que quedaron entre 10 y 12. No pierde tiempo con los de 18 ni con los de 5.

Con eso se consigue casi toda la calidad del modelo caro por una fracción del costo. Es una
idea de ingeniería, no de modelo, y no aparece en ningún tutorial.

---

# Bloque 4 — El momento GPU

## Primero: por qué el benchmark ingenuo miente

> *Analogía:* cronometras cuánto tarda tu mamá en servir la sopa, pero arrancas el
> cronómetro cuando todavía está prendiendo la cocina. El primer plato siempre parece
> lentísimo.

La primera inferencia incluye cargar el modelo a la memoria e inicializar CUDA. Por eso el
protocolo honesto es:

1. **Descartar** las primeras páginas (calentamiento).
2. Medir varias páginas, **las mismas** en todas las condiciones.
3. Reportar la **mediana**, no el promedio: un pico aislado no debe mover la conclusión.
4. Reportar también la **VRAM** y la **calidad**, no solo la velocidad.

El punto 4 importa: una configuración que acelera y empeora la transcripción no es una
mejora.

**Antes de correr: cierra el navegador.** Cada pestaña ocupa VRAM y compite por la GPU.

In [ ]:
# Preparamos un lote de paginas logicas para medir sobre todas lo mismo.
def preparar_lote(ruta_pdf, n_hojas, dpi, rotacion, carpeta="procesadas"):
    Path(carpeta).mkdir(exist_ok=True)
    rutas = []
    doc = pdfium.PdfDocument(str(ruta_pdf))
    total = len(doc)
    doc.close()
    for h in range(min(n_hojas, total)):
        destino = Path(carpeta) / f"h{h:03d}_a.png"
        if not destino.exists():                 # no recomputar lo ya hecho
            img = rasterizar(ruta_pdf, h, dpi)
            inferior, _ = partir_en_dos(img) if CONFIG["dos_por_pagina"] else (img, None)
            inferior.rotate(rotacion, expand=True).save(destino)
        rutas.append(destino)
    return rutas

N = CONFIG["bench_calentamiento"] + CONFIG["bench_medidas"]
LOTE = preparar_lote(RUTA_PDF, N, CONFIG["dpi_fisico"], ROTACION)
print(f"{len(LOTE)} paginas listas para medir")

In [ ]:
import statistics, threading

def medir(motor, rutas, n_calentamiento):
    '''Devuelve mediana de s/pagina y pico de VRAM sobre la linea base.'''
    base = vram_mb()
    pico = {"v": base or 0}
    seguir = {"ok": True}

    def vigilar():
        while seguir["ok"]:
            v = vram_mb()
            if v is not None:
                pico["v"] = max(pico["v"], v)
            time.sleep(0.2)

    hilo = threading.Thread(target=vigilar, daemon=True)
    if base is not None:
        hilo.start()

    for r in rutas[:n_calentamiento]:            # calentamiento: NO se cuenta
        ocr_pagina(motor, r)

    tiempos = []
    for r in rutas[n_calentamiento:]:
        t0 = time.perf_counter()
        ocr_pagina(motor, r)
        tiempos.append(time.perf_counter() - t0)

    seguir["ok"] = False
    delta = (pico["v"] - base) if base is not None else None
    return statistics.median(tiempos), delta

# El lote de CPU es MAS CORTO a proposito, y hay que decirlo en voz alta.
# Medido en la laptop de la clase, a 425 DPI: una pagina tarda ~0.7 s en GPU y
# entre 24 y 68 s en CPU segun el nivel del modelo. Con el mismo lote en las dos
# condiciones, este bloque se come la clase entera. Se reduce la MUESTRA, no el
# protocolo: sigue habiendo calentamiento descartado y sigue siendo la mediana.
LOTE_CPU = LOTE[:CONFIG['bench_calentamiento'] + 2]

filas = []
for dev in (["gpu", "cpu"] if CONFIG["device"] == "gpu" else ["cpu"]):
    lote_dev = LOTE if dev == "gpu" else LOTE_CPU
    print(f"\nMidiendo en {dev.upper()} sobre {len(lote_dev)} paginas ...")
    if dev == "cpu":
        print("  (menos paginas que en GPU: ahi cada una tarda decenas de segundos)")
    motor = MOTOR if dev == CONFIG["device"] else construir_motor(device=dev,
                                                                  idioma=CONFIG["idioma"])
    mediana, vram = medir(motor, lote_dev, CONFIG["bench_calentamiento"])
    filas.append({"device": dev,
                  "mediana_s_pagina": round(mediana, 2),
                  "vram_pico_MB": None if vram is None else round(vram)})

bench = pd.DataFrame(filas)
display(bench)
if len(bench) == 2:
    cpu = bench.loc[bench.device == "cpu", "mediana_s_pagina"].iloc[0]
    gpu = bench.loc[bench.device == "gpu", "mediana_s_pagina"].iloc[0]
    print(f"\nLa GPU fue {cpu/gpu:.1f} veces mas rapida en este documento.")

### Cómo leer esta tabla, en este orden

1. **La VRAM.** En modo GPU sube varios cientos de MB. En modo CPU se queda plana. Si en
   modo GPU no sube, **no estás usando la GPU**, digan lo que digan los otros chequeos.
2. **La mediana.** Debe caer de forma clara.
3. **El texto.** Es **el mismo** en ambos modos. Solo cambia el tiempo.

### Una advertencia honesta

El caso real que originó esta clase midió, sobre una GTX 1660 Ti de 6 GB, unos **45 s por
página en CPU contra ~1 s en GPU**; un rango de 42 páginas pasó de 32 minutos a cerca de 1.

Pero parte de ese 40x es "GPU buena" y parte es "CPU mal configurada". Un CPU con inferencia
optimizada acorta bastante la brecha. Prometer 40x sin matices es vender humo.

---

# Bloque 5 — De texto a datos

Hasta aquí tenemos texto. Lo que un programa necesita son **campos**. Vamos a extraer
cuatro:

| Campo | Valor esperado en el documento real |
|---|---|
| Entidad convocante | Municipalidad Provincial de Ilo |
| Número de procedimiento | Licitación Pública de Obras N° 05-2026-C-MPI |
| Objeto de la contratación | Ampliación del servicio de movilidad urbana, Sector II (PROMUVI IV) |
| Departamento | Moquegua |

Hay dos caminos, y **compararlos es la lección**:

- **Vía A — reglas.** Expresiones regulares. Rápida, gratis, se rompe con una errata del OCR.
- **Vía B — modelo de lenguaje.** Robusta ante erratas, cuesta tokens.

## La regla que no se negocia

> Todo dato extraído debe existir **literalmente** en el texto del OCR. Si no existe, se
> marca como `sin_verificar`.

Un modelo que "corrige" el número de licitación y acierta es tan peligroso como uno que lo
inventa, porque la próxima vez va a corregir y fallar. En documentos legales, la IA es un
**copiloto**, no un piloto automático.

In [ ]:
import unicodedata

# El nombre de la entidad NO esta en la caratula: vive en el encabezado, que pertenece
# a la OTRA pagina logica de la misma hoja. Por eso procesamos las dos mitades.
textos_hoja = []
for k, mitad in enumerate(mitades):
    r = Path(f"procesadas/demo_mitad_{k}.png")
    mitad.rotate(ROTACION, expand=True).save(r)
    textos_hoja.append(" ".join(ocr_pagina(MOTOR, r)["texto"].astype(str).tolist()))

TEXTO_OCR = " ".join(textos_hoja)
print("Caracteres reconocidos en la hoja completa:", len(TEXTO_OCR), "\n")
print(TEXTO_OCR[:700], "...\n")

def normalizar(s):
    '''Minusculas, sin tildes, espacios colapsados. Para comparar sin ser quisquilloso.'''
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).strip().lower()

def verificar_literal(valor, texto):
    '''La regla de oro: el valor debe existir en el texto del OCR.'''
    return normalizar(valor) in normalizar(texto)

REGLAS = {
    "entidad_convocante":   r"(municipalidad\s+provincial\s+de\s+[a-z]+)",
    "numero_procedimiento": r"(licitaci[oó]n\s+p[uú]blica\s+(?:de\s+obras\s+)?n[°ºo]?\s*[\d\-a-z]+)",
    "departamento":         r"departamento\s+de\s+([a-z]+)",
    "objeto_contratacion":  r"(contrataci[oó]n\s+para\s+la\s+ejecuci[oó]n\s+de\s+la\s+obra[:\s].{20,220}?)(?:licitaci|$)",
}

def extraer_por_reglas(texto):
    plano = normalizar(texto)
    salida = {}
    for campo, patron in REGLAS.items():
        m = re.search(patron, plano, re.I | re.S)
        valor = m.group(1).strip() if m else None
        salida[campo] = {"valor": valor,
                         "metodo": "reglas",
                         "verificado": bool(valor) and verificar_literal(valor, texto)}
    return salida

resultado = extraer_por_reglas(TEXTO_OCR)
display(pd.DataFrame(resultado).T)

aciertos = sum(1 for v in resultado.values() if v["valor"])
print(f"\nLa via de reglas ESTRICTAS encontro {aciertos} de 4 campos.")
print("Si encontro menos de 4, ESO es el resultado del experimento, no un error tuyo.")

In [ ]:
# El encabezado de la hoja esta cortado en el escaneo: donde dice PROVINCIAL el OCR
# suele leer ROVINCIAL. Una sola letra tumba la regla. Veamos cuanto cuesta parcharla.
REGLAS_TOLERANTES = dict(REGLAS)
REGLAS_TOLERANTES["entidad_convocante"] = r"(municipalidad\s+\S*rovincial\s+de\s+[a-z]+)"

REGLAS_ORIGINALES = dict(REGLAS)
REGLAS.update(REGLAS_TOLERANTES)
resultado_tolerante = extraer_por_reglas(TEXTO_OCR)
REGLAS.update(REGLAS_ORIGINALES)

comparacion = pd.DataFrame({
    "estricta":  {k: v["valor"] for k, v in resultado.items()},
    "tolerante": {k: v["valor"] for k, v in resultado_tolerante.items()},
})
display(comparacion)
print("Cada errata nueva del escaneo te obliga a parchar otra expresion regular.")
print("A escala, ese parcheo infinito es exactamente el trabajo que un modelo te ahorra.")
print("Esa es la decision de ingenieria, no una preferencia de herramienta.\n")

resultado = resultado_tolerante   # nos quedamos con la version que si encontro los campos

# Entrega: CSV y JSON, como en el proyecto real.
Path("salidas").mkdir(exist_ok=True)

tabla = pd.DataFrame([{"campo": k, **v} for k, v in resultado.items()])
tabla.to_csv("salidas/campos_extraidos.csv", index=False, encoding="utf-8-sig")
df.drop(columns=["caja"]).to_csv("salidas/ocr_lineas.csv", index=False, encoding="utf-8-sig")
with open("salidas/campos_extraidos.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=2)

print("Archivos escritos en salidas/:")
for p in sorted(Path("salidas").iterdir()):
    print(" -", p.name, f"({p.stat().st_size} bytes)")
display(tabla)

## Vía B — con un modelo de lenguaje (opcional)

Solo si tienes acceso a un modelo. **La verificación literal se aplica igual**, y aquí sí
importa: el modelo puede devolver un valor plausible que no está en el documento.

El principio de diseño del proyecto real:

> Pon el trabajo **barato y masivo** en local (el OCR). Reserva el modelo caro para lo que
> de verdad agrega valor (el razonamiento).

In [ ]:
import json, os, sys
from pathlib import Path

# --- La clave NUNCA se escribe aqui -------------------------------------------
# Una clave escrita en una celda se guarda dentro del .ipynb. Si ese .ipynb llega a
# un repositorio publico, la clave queda en el historial de git para siempre y hay
# que revocarla. El proyecto la lee de un .env que .gitignore excluye:
#
#     copy .env.example .env        y pega tu clave dentro del .env
#
# En Colab, donde no hay .env, se pide por teclado con getpass.
raiz = Path.cwd()
while raiz != raiz.parent and not (raiz / "config.yaml").exists():
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

try:
    from src.comun import cargar_entorno
    cargar_entorno()                      # lee el .env si existe
except ImportError:
    pass

if not os.environ.get("ANTHROPIC_API_KEY"):
    import getpass
    escrita = getpass.getpass("Clave de Anthropic (Enter para saltar la via B): ")
    if escrita.strip():
        os.environ["ANTHROPIC_API_KEY"] = escrita.strip()

# USAR_LLM ya no se edita a mano: depende de si hay clave.
USAR_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))
MODELO = "claude-haiku-5-5"   # extraccion simple: el modelo chico alcanza
print("Via B disponible" if USAR_LLM else "Sin clave: solo via A (no necesita API)")
# ------------------------------------------------------------------------------

if USAR_LLM:
    from anthropic import Anthropic

    # El esquema obliga al modelo a devolver JSON valido con estos cuatro campos.
    # Es mas robusto que pedir "responde solo JSON" y luego limpiar los ``` a mano:
    # aqui el formato lo garantiza la API, no la buena voluntad del modelo.
    ESQUEMA = {
        "type": "object",
        "properties": {
            "entidad_convocante":   {"type": ["string", "null"]},
            "numero_procedimiento": {"type": ["string", "null"]},
            "objeto_contratacion":  {"type": ["string", "null"]},
            "departamento":         {"type": ["string", "null"]},
        },
        "required": ["entidad_convocante", "numero_procedimiento",
                     "objeto_contratacion", "departamento"],
        "additionalProperties": False,
    }

    INSTRUCCION = (
        "Extrae los cuatro campos de esta base de licitacion peruana.\n"
        "El texto viene de un OCR sobre un documento escaneado: tiene erratas y "
        "palabras cortadas.\n\n"
        "Reglas:\n"
        "1. Copia los valores TAL COMO APARECEN. No corrijas erratas ni completes "
        "palabras cortadas.\n"
        "2. Si un campo no aparece en el texto, devuelve null. Nunca lo deduzcas.\n"
        "3. No agregues informacion que no este en el texto."
    )

    cliente = Anthropic()
    respuesta = cliente.messages.create(
        model=MODELO,
        max_tokens=1000,
        system=INSTRUCCION,
        messages=[{"role": "user", "content": TEXTO_OCR[:4000]}],
        output_config={"format": {"type": "json_schema", "schema": ESQUEMA}},
    )

    # Con output_config el primer bloque de texto ya es JSON valido: no hay que
    # limpiar ``` ni rezar.
    datos = json.loads(next(b.text for b in respuesta.content if b.type == "text"))

    print("\n--- Resultados Via B ---")
    for campo in REGLAS:
        valor = datos.get(campo)
        # La regla de oro se aplica igual al modelo: el valor debe estar en el OCR.
        ok = valor is not None and verificar_literal(valor, TEXTO_OCR)
        print(f"{campo:<24} {'VERIFICADO' if ok else 'SIN VERIFICAR'}  {valor}")

---

# Bloque 6 — Laboratorio de fallas

Aquí está la diferencia entre esta clase y un tutorial de YouTube: **vas a diagnosticar
leyendo, no adivinando.**

Tres fallas. Te doy el síntoma; tú encuentras la causa.

### Falla 1 — El fallback mudo

Construimos el motor **sin** el parámetro `device`. No hay ningún error. La VRAM no se
mueve. El tiempo se multiplica.

Esto es lo más caro que te puede pasar en un proyecto real: **creer que tienes aceleración y
no tenerla**. Y fíjate que no aparece ninguna advertencia en pantalla. El silencio es el
problema.

In [ ]:
print("Construyendo el motor SIN especificar device ...\n")
antes = vram_mb()
motor_mudo = PaddleOCR(lang=CONFIG["idioma"])   # falta device="gpu"
# Nota: si esta maquina no tiene GPU, PaddleOCR cae a CPU y oneDNN la hace
# reventar. En ese caso la demo no aplica: el fallback mudo solo se puede
# mostrar donde SI hay GPU que desaprovechar.
t0 = time.perf_counter(); ocr_pagina(motor_mudo, LOTE[0]); t_mudo = time.perf_counter()-t0
despues = vram_mb()

print(f"Tiempo: {t_mudo:.2f} s por pagina")
if antes is not None:
    print(f"VRAM: {antes:.0f} -> {despues:.0f} MB (delta {despues-antes:+.0f} MB)")
print("\nNinguna excepcion. Ningun warning. Solo esta lento.")
print("Leccion: no preguntes si la GPU esta activa. MIDELO.")

### Falla 2 — La página girada

Esta sale gratis: el documento real ya viene girado. Solo hay que apagar la corrección de
orientación y mirar la basura que sale.

El caso original reportó exactamente este síntoma:

```
Antes:   SOSORAOOS CS OORSOD...        (ilegible)
Despues: N° CONTRATO | OBJETO | ...    (tabla legible)
```

In [ ]:
# Misma pagina, pero sin enderezar y sin clasificador de orientacion.
torcida = Path("procesadas/pagina_torcida.png")
mitades[0].save(torcida)       # sin rotar

motor_sin_orientacion = construir_motor(device=CONFIG["device"],
                                        idioma=CONFIG["idioma"],
                                        orientacion=False)
df_malo = ocr_pagina(motor_sin_orientacion, torcida)

print("SIN preparar la imagen:")
print("  lineas reconocidas:", len(df_malo))
print("  confianza mediana :", round(df_malo['confianza'].median(), 3) if len(df_malo) else "n/a")
print("  muestra:", df_malo['texto'].head(5).tolist())
print()
print("CON la imagen preparada:")
print("  lineas reconocidas:", len(df))
print("  confianza mediana :", round(df['confianza'].median(), 3) if len(df) else "n/a")
print("  muestra:", df['texto'].head(5).tolist())
print()
print("Mismo modelo. Misma GPU. Lo unico que cambio fue la imagen.")

### Falla 3 — El techo de hardware

El caso original intentó correr un LLM de visión de 3 mil millones de parámetros en una GPU
de 6 GB. El log dijo esto:

```
offloaded 0/37 layers to GPU
compute graph ... size = "6.7 GiB"
total memory  ... size = "10.1 GiB"
```

**Pregunta:** un modelo de 3B en 4 bits pesa unos 2 GB. ¿De dónde salieron los otros 8?

**Respuesta:** de la imagen. Un modelo de visión convierte una página en miles de fragmentos
visuales, y la memoria la dominan esos fragmentos, no los pesos del modelo.

> **Regla práctica:** en modelos de visión para documentos, la VRAM no la decide el tamaño
> del modelo, **la decide la resolución de la página**. Duplicar el DPI puede cuadruplicar
> la memoria.

La siguiente celda lo demuestra sin romperte nada: sube el DPI y mira crecer tiempo y
memoria.

In [ ]:
filas = []
for dpi in [150, 300, 425, 600]:
    img = rasterizar(RUTA_PDF, CONFIG["hoja_demo"], dpi)
    p = partir_en_dos(img)[0].rotate(ROTACION, expand=True)
    ruta = Path(f"procesadas/dpi_{dpi}.png"); p.save(ruta)
    antes = vram_mb()
    t0 = time.perf_counter(); d = ocr_pagina(MOTOR, ruta); seg = time.perf_counter()-t0
    despues = vram_mb()
    filas.append({"dpi": dpi,
                  "megapixeles": round(p.width*p.height/1e6, 1),
                  "segundos": round(seg, 2),
                  "lineas": len(d),
                  "confianza_mediana": round(d["confianza"].median(), 3) if len(d) else None,
                  "vram_MB": None if antes is None else round(despues)})

esc = pd.DataFrame(filas)
display(esc)
print("\nMira dos cosas:")
print(" 1. Los megapixeles crecen con el CUADRADO del DPI. El tiempo los sigue.")
print(" 2. La calidad NO crece igual: a partir de cierto punto subir el DPI solo cuesta.")
print("\nEse punto de quiebre es el DPI que debes poner en tu configuracion.")

---

# Cierre

## Lo que acabas de construir

Los mismos cinco pasos del proyecto real, en escala de juguete:

| Paso | Lo que hiciste | En el proyecto real |
|---|---|---|
| 1. Triaje | clasificaste 57 hojas | decide qué páginas valen OCR en documentos de 500 |
| 2. Preparación | rasterizar, partir, enderezar | es donde se gana la calidad |
| 3. OCR | texto, cajas, confianza | 110 páginas en minutos, en una laptop |
| 4. Medición | CPU vs GPU, DPI | decide si el proyecto es viable |
| 5. Extracción | 4 campos a CSV | alimenta el informe legal |

## Las seis lecciones transferibles

1. **Define el problema** antes de elegir la herramienta.
2. **Conoce tus límites** de hardware: deciden qué es posible.
3. **La herramienta del tamaño del problema.** No traigas un cañón para matar una mosca.
4. **Híbrido:** lo barato y masivo en local, el modelo caro solo donde agrega valor.
5. **Itera y diagnostica.** Los fracasos enseñan; los logs son donde está la respuesta.
6. **Humano al centro.** La IA propone, la persona decide.

## El dato que cierra, porque contradice la intuición

En el ranking público de parsing de documentos de 2026 (OmniDocBench v1.6), los primeros
puestos los tienen modelos de **menos de 1.3 mil millones** de parámetros: PaddleOCR-VL-1.6
con 96.34, MinerU2.5-Pro con 95.75 y GLM-OCR con 95.22. Un modelo de **235 mil millones**
queda en 89.78.

**El pequeño especializado le gana al gigante generalista en su propio terreno.**

Y el segundo dato, el que más importa para documentos escaneados: cuando a estos mismos
modelos se les dan **fotos** de las páginas en vez de escaneos limpios, el mejor cae de
91.93 a 72.19 puntos. **Veinte puntos por calidad de imagen.** No hay modelo que compense
una mala imagen. Por eso el Bloque 2 es el que de verdad importa.

---

## Entregable de la sesión

1. El archivo `salidas/campos_extraidos.csv`.
2. Una línea de respuesta: **¿en qué carril corriste, y cuántos segundos por página te
   tomó?**

La segunda parte es la que cuenta: obliga a haber medido.

---

## Ejercicios para llevar

**Nivel 1.** Cambia `CONFIG["dpi_fisico"]` y encuentra el punto donde subirlo deja de
mejorar la confianza mediana. Justifica tu elección con la tabla, no con la intuición.

**Nivel 2.** Implementa el **escalamiento por confianza**: si una página tiene más del 20%
de líneas bajo 0.80, vuelve a procesarla con un DPI mayor. Reporta cuántas páginas
escalaron y cuánto mejoró la confianza promedio.

**Nivel 3.** Escribe un **detector automático de dos páginas por hoja**. Pista: compara la
densidad de tinta de la mitad superior y la inferior, y busca la franja blanca del centro.
Esto no existe en ninguna librería; si lo resuelves bien, es material de innovación para tu
proyecto final.

**Nivel 4.** Toma cualquier documento público peruano escaneado (SEACE, El Peruano, una
resolución municipal) y corre este notebook completo sobre él. Reporta qué se rompió y por
qué. *Que algo se rompa es el resultado esperado: ahí es donde se aprende.*